In [ ]:
import os 
import sys
import ipynbname
import gradio as gr
import json
import sass 
import sys
from src.logger import LOGGER

root = ipynbname.path().parents[1]
sys.path.insert(0, str(root))

# tandem folder
TANDEM_DIR = os.path.join(root, 'tandem')
GRADIO_DIR = os.path.join(root, 'gradio_app')
TMP_DIR = os.path.join(GRADIO_DIR, 'tmp')
JOB_DIR = os.path.join(TANDEM_DIR, 'jobs')

SASS_DIR = os.path.join(GRADIO_DIR, "sass")
ASSETS_DIR = os.path.join(GRADIO_DIR, "assets")

figure_1 = os.path.join(ASSETS_DIR, 'images/figure_1.jpg')

sass.compile(dirname=(str(SASS_DIR), str(ASSETS_DIR)), output_style="expanded")
sys.path.append(TANDEM_DIR)
with open(os.path.join(ASSETS_DIR, "main.css")) as f:
    custom_css = f.read()

root, TMP_DIR, gr.__version__, os.getcwd()

In [1]:
import json
file = '/home/loci/main/tandem_website_dev/tandem/jobs/test/test_website/user_log.jsonl'
events = []
with open(file, "r", encoding="utf-8") as handle:
    for line in handle:
        line = line.strip()
        if not line:
            continue
        try:
            events.append(json.loads(line))
        except json.JSONDecodeError:
            continue

In [ ]:

STAGE_LABELS = [
    "Validating input",
    "Mapping SAVs to structures",
    "Feature calculation",
    "Model inferencing/Training",
    "Generating report",
]
def _build_stage_cells(mode, events, job_folder, job_status):
    stage_cells = {}
    for i, label in enumerate(STAGE_LABELS, start=1):
        event = next((event for event in events if (event['stage']==label) and (event['level']=='info')), None)

        if not event:
            stage_cells[f"status_{i}"] = f'<span>{html.escape('pending')}</span>'
            stage_cells[f"file_{i}"] = ""
            stage_cells[f"time_{i}"] = ""

        stage_cells[f"time_{i}"] = html.escape(event.get("context").get("duration_text", ""))
        file_text = _build_file_links(job_folder, event)

        if event is not None:
            status = "Done"
        elif job_status == "processing":
            status = "Run"
        else:
            status = "Pend"
        stage_cells[f"status_{i}"] = f'<span>{html.escape(status)}</span>'
    return stage_cells

results = {
    "status_0": "Done",
    "file_0": "SAVs.txt",
    "time_0": "0.2 s",

    "status_0": "Done (2 Warning, 2 Error)",
    "file_0": "Uniprot2PDB.txt",
    "time_0": "0.2 s",
    
    ...
}

In [4]:

STAGE_LABELS = [
    "Validating input",
    "Mapping SAVs to structures",
    "Feature calculation",
    "Model inferencing/Training",
    "Generating report",
]
def _build_stage_cells(events, job_status):
    results = {}

    for i, label in enumerate(STAGE_LABELS, start=1):
        main_event = next((event for event in reversed(events) if event.get("stage") == label and event.get("level") == "info"),None)

        n_warning = sum(1 for event in events if event.get("stage") == label and event.get("level") == "warning")
        n_error = sum(1 for event in events if event.get("stage") == label and event.get("level") == "error")

        if main_event is None:
            # If pending job, status: "Pend"
            # IF processing job, status depends previous event. 
            # If previous event is Done, status is "Process", else "Pend"
            if job_status == "processing":
                status = "Process" if "Done" in results[f"status_{i-1}"] else "Pend"
            else:
                status = "Pend"
            results[f"file_{i}"] = ""
            results[f"time_{i}"] = ""
        else:
            status = "Done"
            context = main_event.get("context", {})
            results[f"file_{i}"] = str(context.get("file", ""))
            results[f"time_{i}"] = str(context.get("duration_text", ""))

        parts = []
        if n_warning:
            parts.append(f"{n_warning} Warning")
        if n_error:
            parts.append(f"{n_error} Error")

        results[f"status_{i}"] = f"{status} ({', '.join(parts)})" if parts else status
    return results

_build_stage_cells(events, 'processing')

{'file_1': 'SAVs.txt',
 'time_1': '0.2 s',
 'status_1': 'Done',
 'file_2': 'Uniprot2PDB.txt',
 'time_2': '21.2 s',
 'status_2': 'Done',
 'file_3': 'features.csv',
 'time_3': '3.9 min',
 'status_3': 'Done',
 'file_4': 'Main_Predictions.csv',
 'time_4': '13.7 s',
 'status_4': 'Done',
 'file_5': 'log.txt',
 'time_5': '0.0 s',
 'status_5': 'Done'}